# Kelly Criterion Explorer

Interactive analysis of Kelly variants: binary, fractional, multi-outcome, correlated, and uncertain-edge.
All charts rendered by the shared `ui.core.charts` engine.

In [ ]:
import numpy as np
from ipywidgets import FloatSlider, FloatLogSlider, interactive, HBox, VBox, Output
from IPython.display import display

from speculator_ev_engine.core.kelly import (
    binary_kelly, fractional_kelly, multi_outcome_kelly, uncertain_edge_kelly,
)
from speculator_ev_engine.core.bankroll import kelly_fraction_sweep, simulate_bankroll
from speculator_ev_engine.ui.core.charts import kelly_fraction_vs_growth, uncertain_edge_kelly as uncertain_edge_chart
from speculator_ev_engine.ui.core.formatters import fmt_ev, fmt_frac, fmt_ruin

## 1. Binary Kelly: Fraction vs Growth Rate

In [ ]:
out = Output()

def update_kelly(p, b):
    with out:
        out.clear_output(wait=True)
        sweep = kelly_fraction_sweep(p, b, n_steps=100, n_simulations=2000, seed=42)
        fracs = np.array([s[0] for s in sweep])
        growths = np.array([s[1] for s in sweep])
        ruins = np.array([s[2] for s in sweep])
        opt_idx = int(np.argmax(growths))
        fig = kelly_fraction_vs_growth(fracs, growths, ruins, optimal_idx=opt_idx)
        display(fig)
        plt.close(fig)

p_slider = FloatSlider(value=0.55, min=0.51, max=0.75, step=0.01, description='Win P')
b_slider = FloatSlider(value=1.0, min=0.5, max=5.0, step=0.1, description='Odds')
w = interactive(update_kelly, p=p_slider, b=b_slider)
display(VBox([HBox([p_slider, b_slider]), out]))

## 2. Uncertain Edge: Kelly Shrinks with Uncertainty

In [ ]:
out2 = Output()

def update_uncertain(edge_mean, edge_std):
    with out2:
        out2.clear_output(wait=True)
        k = uncertain_edge_kelly(edge_mean, edge_std, odds=1.0, n_samples=5000, seed=42)
        samples = []
        rng = np.random.default_rng(42)
        edges = rng.normal(edge_mean, edge_std, 5000)
        for e in edges:
            p = (1.0 + e) / 2.0
            p = np.clip(p, 0.001, 0.999)
            samples.append(binary_kelly(p, 1.0).fraction)
        fig = uncertain_edge_chart(np.array(samples), k.fraction)
        display(fig)
        plt.close(fig)

em_slider = FloatSlider(value=0.05, min=0.0, max=0.15, step=0.005, description='Edge μ')
es_slider = FloatSlider(value=0.02, min=0.0, max=0.10, step=0.005, description='Edge σ')
w2 = interactive(update_uncertain, edge_mean=em_slider, edge_std=es_slider)
display(VBox([HBox([em_slider, es_slider]), out2]))